В данном файле выполнен код для парсинга юрдических статей с сайта https://pravo.ru/ по следующему плану:

*   Парсим тэги страниц с темами статей;  
*   Для каждой темы парсим ссылки на страницы с аннотациями статей;
*   С каждой страницы парсим ссылки на страницы со статьями;
*   Проходим по каждой ссылке со статьей и парсим текст статьи, краткое содержание, заголовок, текст статьи, категорию.

In [ ]:
from re import L
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from tqdm import tqdm

1. Парсим тэги тем.

In [ ]:
def get_tags(base_url):
  sites=[base_url]
  current_link=base_url
  site=requests.get(current_link)
  soup=BeautifulSoup(site.text)

  tags={}

  theme=soup.find('section', class_='column medium-3')

  for topic in theme.find_all('a', class_=''):
    category=topic.text
    tags[category]=topic.get('href')
  return tags
tags=get_tags('https://pravo.ru/')

2. Парсим ссылки на страницы с аннотациями статей и сохраним их в файл all_pages.csv

In [ ]:
def get_pages(tag):
  base_url=tag
  current_link=base_url
  pages=[base_url]

  page=1
  while True:
    page+=1
    site=requests.get(current_link)
    soup=BeautifulSoup(site.text)
    row=soup.find('ul', class_='pagination column')
    found=False
    for next_link in row.find_all('a'):
      if next_link:
        if str(page) in next_link.text:
          next_url=next_link.get('href')
          found=True
          pages.append(next_url)
          current_link=next_url
    if found==False:
      break
  return pages
all_pages={}
for cat in tags:
  all_pages[cat]=get_pages(tags[cat])
print(all_pages)

In [ ]:
df = pd.DataFrame.from_dict(all_pages, orient='index').T
df.to_csv('all_pages.csv', index=False, encoding='utf-8')

3. Парсим ссылки на страницы с тестами статей. Сохраним ссылки в файле all_links_to_articles.csv

In [ ]:
df=pd.read_csv('all_pages.csv')
df.fillna('', inplace=True)

In [ ]:
def get_articles(base_url):
  sites=[base_url]
  current_link=base_url
  site=requests.get(current_link)
  soup=BeautifulSoup(site.text)

  articles=[]

  tablo=soup.find('div', class_='row small-up-12 tiles_row')
  for article in tablo.find_all('div', class_='column'):
    if article:
      col=article.find('a', href=True, class_=False)
      if col:
        art=col.get('href')
        articles.append(art)
  return articles

articles={}

request_count=0
for category in df:
  articles[category]=[]
  for site in df[category]:
    if site!='':
      request_count += 1
      if request_count % 10 == 0:
        time.sleep(3)

      article=get_articles(site)
      articles[category].append(article)
      print(site)

      #time.sleep(1)

print(articles)

In [ ]:
df.to_csv('all_links_to_articles.csv', index=False, encoding='utf-8')

4. Проходим по каждой ссылке на статью и парсим текста, залоговок, краткое содержание статей. Сохраняем в файле all_article.csv

In [ ]:
df=pd.read_csv('all_links_to_articles.csv')
df.fillna('', inplace=True)

In [ ]:
def get_text(base_url, headers=None):
    if headers is None:
        headers = {
            'User-Agent': random.choice([
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0',
                'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.15'
            ])
        }

    sites = [base_url]
    current_link = base_url

    for attempt in range(3):
        try:
            response = requests.get(base_url, headers=headers, timeout=10)

            if response.status_code == 404:
                return ['', '', '']

            if response.status_code != 200:
                print(f"HTTP {response.status_code} for: {base_url}")
                if attempt == 2:
                    return ['', '', '']
                time.sleep(2)
                continue

            soup = BeautifulSoup(response.text, 'html.parser')

            title = soup.find('h1', class_='block_header')
            if title:
                title = title.text
            else:
                title = ''

            short_description = soup.find('section', class_='lead')
            if short_description:
                short_description = short_description.text
            else:
                short_description = ''

            def clean_text(text):
                if text:
                    text = text.replace('\n', ' ').replace('\t', ' ')
                    text = ' '.join(text.split())
                    return text.strip()
                return ''

            article = ''
            full_text = soup.find('section', class_='full_text clearfix')
            if full_text:
                for p in full_text.find_all('p'):
                    if p:
                        for span in p.find_all('span'):
                            span.decompose()
                        article += p.text + ' '

            article = clean_text(article)

            return [
                clean_text(title),
                clean_text(short_description),
                article
            ]

        except requests.exceptions.ReadTimeout:
            print(f"Attempt {attempt + 1} timeout for: {base_url}")
            if attempt == 2:
                return ['', '', '']
            time.sleep(2)

        except requests.exceptions.RequestException as e:
            print(f"Request error for {base_url}: {str(e)}")
            if attempt == 2:
                return ['', '', '']
            time.sleep(2)

    return ['', '', '']

In [ ]:
count = 0
request_count = 0

BASE_DELAY = 2.0
RANDOM_DELAY_RANGE = (1.0, 4.0)
LONG_PAUSE_INTERVAL = random.randint(20, 40)
LONG_PAUSE_DURATION = (15, 30)

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.15'
]

for i, article_url in tqdm(enumerate(df['article_site'])):
    try:
        headers = {'User-Agent': random.choice(USER_AGENTS)}

        result = get_text(article_url, headers=headers)

        mask = df['article_site'] == article_url
        df.loc[mask, ['title', 'short_description', 'article']] = result
        request_count += 1
        delay = BASE_DELAY + random.uniform(*RANDOM_DELAY_RANGE)
        time.sleep(delay)

        if request_count % LONG_PAUSE_INTERVAL == 0:
            long_pause = random.uniform(*LONG_PAUSE_DURATION)
            print(f"Пауза {long_pause:.1f} сек после {request_count} запросов")
            time.sleep(long_pause)

        if random.random() < 0.05:
            extra_pause = random.uniform(5, 15)
            print(f"Случайная пауза {extra_pause:.1f} сек")
            time.sleep(extra_pause)

    except Exception as e:
        print(f"Ошибка: {article_url}: {e}")
        error_pause = 10 + (5 * count)
        print(f"Пауза {error_pause} сек из-за ошибки")
        time.sleep(error_pause)
        count += 1

        if count >= 3:
            print("Много ошибок! Пауза 60 сек")
            time.sleep(60)
            count = 0

In [ ]:

df.to_csv('all_articles.csv', index=False, encoding='utf-8')

Вывод:
В результате парсинга собрано около 18 тысяч статей на юридическую тематику с сайта pravo.ru с тематической разметкой.